In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/adn_boca_real_features.csv')

print(f'Dataset: {df.shape}')
print(df.isnull().sum())

In [ ]:
le = LabelEncoder()
df['posicion_encoded'] = le.fit_transform(df['posicion'])

# rating y rendimiento NO se usan: etiqueta se define como
# (rating >= 7.0) & (goles + asistencias > 2) -> data leakage directo
features = [
    'posicion_encoded', 'edad', 'partidos',
    'goles', 'asistencias', 'pases_precisos',
    'goles_por_partido', 'participacion_gol', 'experiencia'
]

X = df[features]
y = df['etiqueta']

print('Distribucion de etiquetas:')
print(y.value_counts())
print(f'\nFeatures ({len(features)}): {features}')
print(f'Clase minoritaria: {y.mean():.1%} del dataset')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train ratio: {y_train.mean():.1%} | Test ratio: {y_test.mean():.1%}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print('='*50)
print('RANDOM FOREST (regularizado)')
print('='*50)
print(classification_report(y_test, y_pred_rf,
      target_names=['No encaja', 'Perfil Boca']))
print('\nMatriz de confusion:')
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        l1_ratio=1,
        solver='saga',
        C=0.1,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print('='*50)
print('LOGISTIC REGRESSION (L1 / Lasso)')
print('='*50)
print(classification_report(y_test, y_pred_lr,
      target_names=['No encaja', 'Perfil Boca']))
print('\nMatriz de confusion:')
print(confusion_matrix(y_test, y_pred_lr))

coefs = lr.named_steps['clf'].coef_[0]
print('\nCoeficientes L1 (0 = feature descartada):')
for f, c in sorted(zip(features, coefs), key=lambda x: -abs(x[1])):
    print(f'  {f:25s} {c:+.4f}')

In [ ]:
from sklearn.model_selection import cross_val_score

print('VALIDACION CRUZADA (5 folds)')
print('='*50)

for name, model in [('RandomForest', rf), ('Logistic L1', lr)]:
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'{name:15s} | CV Accuracy: {scores.mean():.3f} (+/- {scores.std()*2:.3f})')
    scores_f1 = cross_val_score(model, X, y, cv=5, scoring='f1')
    sep = ''
    print(f'{sep:15s} | CV F1:      {scores_f1.mean():.3f} (+/- {scores_f1.std()*2:.3f})')

In [ ]:
import matplotlib.pyplot as plt

importancias = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot(kind='barh', ax=ax, color='#003087')
ax.set_title('Importancia de Features - Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print('\nImportancia de cada feature:')
for f, v in importancias.items():
    print(f'  {f:25s} {v:.4f}')

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(rf, '../models/modelo_adn_boca.pkl')
joblib.dump(lr, '../models/modelo_logistic_l1.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

print('Modelos guardados:')
print('  models/modelo_adn_boca.pkl      (RandomForest)')
print('  models/modelo_logistic_l1.pkl   (Logistic L1)')
print('  models/label_encoder.pkl')

In [ ]:
# Scoring sobre todo el dataset con ambos modelos
df_score = df.copy()

df_score['probabilidad_rf'] = rf.predict_proba(X)[:, 1]
df_score['pred_rf'] = rf.predict(X)

df_score['probabilidad_lr'] = lr.predict_proba(X)[:, 1]
df_score['pred_lr'] = lr.predict(X)

top_rf = df_score[df_score['pred_rf'] == 1].sort_values('probabilidad_rf', ascending=False)
top_lr = df_score[df_score['pred_lr'] == 1].sort_values('probabilidad_lr', ascending=False)

print('TOP 10 - Random Forest')
print('='*60)
print(top_rf[['nombre', 'posicion', 'temporada', 'goles', 'asistencias', 'probabilidad_rf']].head(10).to_string(index=False))

print('\nTOP 10 - Logistic L1')
print('='*60)
print(top_lr[['nombre', 'posicion', 'temporada', 'goles', 'asistencias', 'probabilidad_lr']].head(10).to_string(index=False))